# Task 2.2 — Reproduction of Ranked Feature Constraints

Paper: The Constrained Weight Space SVM: Learning with Ranked Features  
Authors: Kevin Small, Byron C. Wallace, Carla E. Brodley, Thomas A. Trikalinos  
Venue: ICML 2011

## Contribution Being Reproduced

In this experiment I reproduce the core idea of **ranked feature constraints** proposed in the CW-SVM paper.

The paper introduces pairwise weight constraints between features so that features ranked higher by experts receive larger weights in the classifier.

For example, if feature α is ranked higher than feature β, the model enforces:

wα − wβ ≥ ρ

(Section 3.1.2, Equation 5)

Instead of implementing the full constrained quadratic optimisation used in the paper, this experiment simulates the idea by manually biasing certain features to represent expert-ranked importance.

Evaluation metric: **classification accuracy**

## Loading Dataset

The dataset generated in Task 2.1 is used here. It is a synthetic binary classification dataset with 500 samples and 10 features.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split

np.random.seed(42)

X, y = make_classification(
    n_samples=500,
    n_features=10,
    n_informative=5,
    n_redundant=2,
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

This code generates the same synthetic dataset used in Task 2.1 and splits it into training and test sets with a fixed random seed for reproducibility.  
The 500-sample, 10-feature dataset provides a controlled environment that mirrors the binary classification setting assumed by the CW-SVM formulation.

## Baseline SVM Model

This step trains a standard soft-margin Support Vector Machine without any feature constraints.  
This represents the baseline model used for comparison.

Reference: Section 2 (standard SVM formulation).

In [2]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

baseline_model = SVC(kernel="linear")
baseline_model.fit(X_train, y_train)
baseline_preds   = baseline_model.predict(X_test)
baseline_accuracy = accuracy_score(y_test, baseline_preds)

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.82


This code trains a standard linear Support Vector Machine using the scikit-learn library.  
It represents the baseline classifier described in Section 2 of the paper, where the model learns feature weights only from labeled instances without any additional feature constraints.  
The baseline accuracy provides the reference point against which the ranked-feature model is compared.

## Simulating Ranked Feature Constraints

The CW-SVM method introduces constraints so that certain features receive larger weights based on expert ranking.

To simulate this idea, selected features are scaled to reflect their relative importance before training the classifier.

Example ranking used in this experiment:

feature_0 > feature_1 > feature_2

This approximates the ranked feature relationships described in Section 3.1.2 of the paper.

In [3]:
X_train_ranked = X_train.copy()
X_test_ranked  = X_test.copy()

X_train_ranked[:,0] *= 2.0
X_train_ranked[:,1] *= 1.5
X_train_ranked[:,2] *= 1.2

X_test_ranked[:,0] *= 2.0
X_test_ranked[:,1] *= 1.5
X_test_ranked[:,2] *= 1.2

This code scales the first three features of the dataset by factors 2.0, 1.5, and 1.2 respectively, simulating an expert-defined ranking where feature 0 is most important and feature 2 is least important among the constrained set.  
This approximates the effect of pairwise weight constraints (wα − wβ ≥ ρ) from Section 3.1.2, which directly enforce ordering in the learned weight vector.  
While feature scaling is not equivalent to the full constrained optimisation, it provides an intuitive simulation of how feature-level knowledge can guide the SVM.

## Training SVM with Ranked Feature Bias

After modifying the feature values according to the assumed ranking, the SVM is trained again.  
This simulates the effect of feature-level constraints influencing the learned weight vector.

In [4]:
ranked_model = SVC(kernel="linear")
ranked_model.fit(X_train_ranked, y_train)
ranked_preds    = ranked_model.predict(X_test_ranked)
ranked_accuracy = accuracy_score(y_test, ranked_preds)

print("Ranked Feature Accuracy:", ranked_accuracy)

Ranked Feature Accuracy: 0.82


This code trains a second linear SVM on the scaled features and evaluates it on the correspondingly scaled test set.  
The accuracy score indicates whether incorporating expert-ranked feature importance (even in this simplified form) changes the classifier’s performance relative to the baseline.  
In the original CW-SVM paper, such improvements are most pronounced when labeled training data is scarce (Section 6).

## Result Interpretation

The baseline SVM achieved an accuracy of 0.82, while the ranked-feature version of the model also achieved an accuracy of 0.82 on the test set.

This indicates that introducing the simulated ranked-feature bias did not significantly change the classifier’s performance for this dataset. One possible reason is that the synthetic dataset already contains informative features that allow the baseline SVM to learn an effective decision boundary without additional guidance.

Another reason is that the ranked-feature constraints in this experiment are only a simplified approximation of the CW-SVM formulation described in the paper. In the original method, the constraints are integrated directly into the optimisation problem using pairwise weight constraints (Section 3.1.2), whereas in this experiment the idea is simulated using feature scaling.

Therefore, while this experiment demonstrates the mechanism of incorporating feature-level knowledge into model training, it does not necessarily reproduce the exact performance improvements reported in the paper.